<a href="https://colab.research.google.com/github/hopeof-Greatmind/NLP_Basic/blob/main/Emotion_Inference_KoBert_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 의존성 패키지 설치 (Colab 셀에서 실행)

In [28]:
!pip install -q transformers==4.0.0 torch
!pip install -q 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
  Preparing metadata (setup.py) ... done


# 2. Module import

In [29]:
import torch
import torch.nn.functional as F
from transformers import BertForSequenceClassification
from kobert_tokenizer import KoBERTTokenizer
from transformers import AutoTokenizer

# 2. 하이퍼파라미터 및 디바이스 설정

In [32]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_LABELS = 2  # 추출할 감정 클래스 수 (예: 2=긍/부정, 6=다중 감정)
MODEL_NAME = 'skt/kobert-base-v1'

# 3. Tokenizer 및 Model 로드
# 주의: SKT KoBERT는 전용 토크나이저(kobert_tokenizer) 사용이 필수적입니다.

In [33]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.to(DEVICE)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: skt/kobert-base-v1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(8002, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, 

# 4. 추론(Inference) 함수 정의

In [35]:
def extract_emotion(text: str) -> dict:
    """
    입력된 텍스트의 감정 클래스 및 확률 분포를 추출합니다.
    """
    model.eval()

    # 전처리 및 토큰화
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

        # Softmax를 통한 확률 산출
        probabilities = F.softmax(logits, dim=-1).squeeze()
        predicted_class = torch.argmax(probabilities, dim=-1).item()

    return {
        "text": text,
        "predicted_class": predicted_class,
        "probabilities": probabilities.cpu().numpy().tolist()
    }

# 5. 테스트 실행

In [36]:
if __name__ == "__main__":
    sample_text = "매우 어렵습니다."

    # Note: 아래 결과는 Fine-tuning 전 무작위 가중치에 의한 결과이므로,
    # 실제 활용을 위해서는 NSMC 또는 AI Hub 감성 대화 말뭉치 등을 이용한 학습이 필요합니다.
    result = extract_emotion(sample_text)

    print(f"Input Text: {result['text']}")
    print(f"Predicted Class: {result['predicted_class']}")
    print(f"Class Probabilities: {[round(p, 4) for p in result['probabilities']]}")

Input Text: 매우 어렵습니다.
Predicted Class: 1
Class Probabilities: [0.38, 0.62]
